# SciERC NER — Experiments

| # | Model | Backbone | Head |
|---|-------|----------|------|
| 1 | `bert_linear` | bert-base-cased | Linear |
| 2 | `scibert_linear` | allenai/scibert_scivocab_cased | Linear |
| 3 | `scibert_mlp` | SciBERT | MLP (768→256→13, GELU) |
| 4 | `scibert_crf` | SciBERT | Linear + CRF |
| 5 | `scibert_mlp_crf` | SciBERT | MLP + CRF |
| 6 | `deberta_qlora` | microsoft/deberta-v3-large | QLoRA (4-bit, r=16) + Linear |

Результаты пишутся в `results/{model_name}.json`. Последние ячейки строят графики.

---
## 0. Setup

In [ ]:
!pip install -q transformers torch seqeval peft bitsandbytes torchcrf

In [ ]:
import sys, os, urllib.request, tarfile, shutil
from pathlib import Path

# src/ импортируется как пакет из корня репозитория
ROOT = Path(os.getcwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR    = ROOT / "data"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Скачиваем SciERC, если данных ещё нет
if not (DATA_DIR / "train.json").exists():
    DATA_DIR.mkdir(exist_ok=True)
    print("Downloading SciERC data...")
    urllib.request.urlretrieve(
        "https://nlp.cs.washington.edu/sciIE/data/sciERC_processed.tar.gz",
        ROOT / "scierc.tar.gz",
    )
    with tarfile.open(ROOT / "scierc.tar.gz", "r:gz") as tar:
        tar.extractall(ROOT)
    for f in (ROOT / "processed_data" / "json").glob("*.json"):
        shutil.copy(f, DATA_DIR / f.name)
    print("Data ready:", [p.name for p in DATA_DIR.glob("*.json")])
else:
    print("Data already present:", [p.name for p in DATA_DIR.glob("*.json")])

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# Общий хелпер run() — перезапустить при рестарте ядра
from torch.optim import AdamW
from torch.utils.data import DataLoader

from src.config import ExperimentConfig
from src.data import build_datasets, ID2LABEL, NUM_LABELS, ENTITY_TYPES
from src.models import build_model
from src.train import train_model, evaluate
from src.utils import set_seed, save_results, load_results


def run(cfg: ExperimentConfig) -> dict:
    set_seed(cfg.seed)
    print(f"\n{'='*56}\n  {cfg.model_name}  ({cfg.base_model})\n{'='*56}")

    train_ds, dev_ds, test_ds = build_datasets(DATA_DIR, cfg.base_model, cfg.max_length)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    dev_loader   = DataLoader(dev_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = build_model(cfg.model_name, cfg.base_model, NUM_LABELS,
                        cfg.use_qlora, cfg.lora_rank, cfg.lora_alpha)
    if not cfg.use_qlora:
        model = model.to(DEVICE)

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.lr, weight_decay=0.01,
    )

    _, history = train_model(model, train_loader, dev_loader, optimizer,
                             DEVICE, cfg.num_epochs, cfg.model_name)

    test_metrics = evaluate(model, test_loader, DEVICE, ID2LABEL)

    result = {
        "model_name":   cfg.model_name,
        "base_model":   cfg.base_model,
        "config":       cfg.__dict__,
        "history":      history,
        "test_metrics": test_metrics,
    }
    save_results(result, RESULTS_DIR / f"{cfg.model_name}.json")

    print(f"\nTest macro F1: {test_metrics['macro_f1']:.4f}")
    for et in ENTITY_TYPES:
        print(f"  {et:<24} {test_metrics.get(et+'_f1', 0):.4f}")
    return result


print("Helper ready.")

---
## Experiment 1 — BERT-base-cased + Linear

In [ ]:
r1 = run(ExperimentConfig(
    model_name="bert_linear",
    base_model="bert-base-cased",
    lr=2e-5,
    batch_size=16,
    num_epochs=10,
    seed=42,
))

---
## Experiment 2 — SciBERT + Linear

In [ ]:
r2 = run(ExperimentConfig(
    model_name="scibert_linear",
    base_model="allenai/scibert_scivocab_cased",
    lr=2e-5,
    batch_size=16,
    num_epochs=10,
    seed=42,
))

---
## Experiment 3 — SciBERT + MLP (768 → 256 → 13, GELU)

In [ ]:
r3 = run(ExperimentConfig(
    model_name="scibert_mlp",
    base_model="allenai/scibert_scivocab_cased",
    lr=2e-5,
    batch_size=16,
    num_epochs=10,
    seed=42,
))

---
## Experiment 4 — SciBERT + CRF

In [ ]:
r4 = run(ExperimentConfig(
    model_name="scibert_crf",
    base_model="allenai/scibert_scivocab_cased",
    lr=2e-5,
    batch_size=16,
    num_epochs=10,
    seed=42,
))

---
## Experiment 5 — SciBERT + MLP + CRF

In [ ]:
r5 = run(ExperimentConfig(
    model_name="scibert_mlp_crf",
    base_model="allenai/scibert_scivocab_cased",
    lr=2e-5,
    batch_size=16,
    num_epochs=10,
    seed=42,
))

---
## Experiment 6 — DeBERTa-v3-large + QLoRA (4-bit, r=16, α=32)

In [ ]:
r6 = run(ExperimentConfig(
    model_name="deberta_qlora",
    base_model="microsoft/deberta-v3-large",
    lr=2e-5,
    batch_size=8,
    num_epochs=10,
    seed=42,
    use_qlora=True,
    lora_rank=16,
    lora_alpha=32,
))

---
## Results & Visualisation

Загружает все `results/*.json` и рисует:
1. Bar — macro F1 по моделям
2. Grouped bar — F1 по каждому типу сущностей
3. Learning curves — dev macro F1 по эпохам
4. Loss curves — train loss по эпохам

In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path("results")
ENTITY_TYPES = ["Generic", "Material", "Method", "Metric", "OtherScientificTerm", "Task"]
MODEL_ORDER  = ["bert_linear", "scibert_linear", "scibert_mlp",
                "scibert_crf", "scibert_mlp_crf", "deberta_qlora"]
LABELS = {
    "bert_linear":     "BERT\nLinear",
    "scibert_linear":  "SciBERT\nLinear",
    "scibert_mlp":     "SciBERT\nMLP",
    "scibert_crf":     "SciBERT\nCRF",
    "scibert_mlp_crf": "SciBERT\nMLP+CRF",
    "deberta_qlora":   "DeBERTa\nQLoRA",
}

results = {}
for name in MODEL_ORDER:
    p = RESULTS_DIR / f"{name}.json"
    if p.exists():
        with open(p) as f:
            results[name] = json.load(f)

models = [m for m in MODEL_ORDER if m in results]
print(f"Loaded: {models}")

# Сводная таблица
hdr = f"{'Model':<22}  {'MacroF1':>8}" + "".join(f"  {e[:8]:>8}" for e in ENTITY_TYPES)
print("\n" + hdr)
print("-" * len(hdr))
for name in models:
    m = results[name]["test_metrics"]
    row = f"{name:<22}  {m['macro_f1']:>8.4f}"
    row += "".join(f"  {m.get(e+'_f1', 0):>8.4f}" for e in ENTITY_TYPES)
    print(row)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

PALETTE = plt.cm.tab10(np.linspace(0, 0.6, 6))
colors  = [PALETTE[i] for i in range(len(models))]

macro_f1s = [results[m]["test_metrics"]["macro_f1"] for m in models]
x_labels  = [LABELS.get(m, m) for m in models]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(x_labels, macro_f1s, color=colors, edgecolor="white", linewidth=0.8, zorder=3)
for bar, val in zip(bars, macro_f1s):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_ylabel("Macro F1 (test)", fontsize=11)
ax.set_title("SciERC NER — Macro F1 by Model", fontsize=13, fontweight="bold")
ax.set_ylim(0, min(1.0, max(macro_f1s) * 1.15 + 0.02))
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "macro_f1_comparison.png", dpi=150)
plt.show()

In [ ]:
n_models   = len(models)
n_entities = len(ENTITY_TYPES)
x     = np.arange(n_entities)
width = 0.75 / n_models

fig, ax = plt.subplots(figsize=(13, 5))
for i, model in enumerate(models):
    vals   = [results[model]["test_metrics"].get(e + "_f1", 0) for e in ENTITY_TYPES]
    offset = (i - n_models / 2 + 0.5) * width
    ax.bar(x + offset, vals, width=width * 0.9,
           label=LABELS.get(model, model).replace("\n", " "),
           color=colors[i], edgecolor="white", linewidth=0.4, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(ENTITY_TYPES, fontsize=10)
ax.set_ylabel("F1 (test)", fontsize=11)
ax.set_title("SciERC NER — Per-Entity F1 by Model", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "entity_f1_comparison.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for i, model in enumerate(models):
    history = results[model]["history"]
    epochs  = [h["epoch"]        for h in history]
    dev_f1s = [h["dev_macro_f1"] for h in history]
    ax.plot(epochs, dev_f1s, marker="o", markersize=4, linewidth=1.8,
            label=LABELS.get(model, model).replace("\n", " "), color=colors[i])

ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Dev Macro F1", fontsize=11)
ax.set_title("SciERC NER — Dev Macro F1 Learning Curves", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
ax.grid(linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "learning_curves.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for i, model in enumerate(models):
    history = results[model]["history"]
    epochs = [h["epoch"]      for h in history]
    losses = [h["train_loss"] for h in history]
    ax.plot(epochs, losses, marker="o", markersize=4, linewidth=1.8,
            label=LABELS.get(model, model).replace("\n", " "), color=colors[i])

ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Train Loss", fontsize=11)
ax.set_title("SciERC NER — Training Loss Curves", fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=9, framealpha=0.9)
ax.grid(linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "loss_curves.png", dpi=150)
plt.show()